<a href="https://colab.research.google.com/github/biglalo104/Projects/blob/main/CNN_Implementation_and_Framework_Comparison_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [22]:
# Install the necessary libararies
import time
import numpy as np
import matplotlib.pyplot as plt

In [23]:
#Import TensorFlow
import tensorflow as tf
from tensorflow.keras import layers,models

In [24]:
#Import PyTorch
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torchvision import datasets,transforms

In [25]:
# Set random seeds for reproducibility
tf.random.set_seed(42)
torch.manual_seed(42)
np.random.seed(42)

print("Libraries imported successfully!")

Libraries imported successfully!


In [26]:
#Load CIFOR-10 Data in TensorFlow
#1.Download and load the dataset
(train_images,train_labels),(test_images,test_labels) = tf.keras.datasets.cifar10.load_data()

#2.Normalize the pixel values to be between 0 amd 1
#(Neural networks perform much better with scaled inputs)
train_images,test_images = train_images/255.0,test_images/255.0

#3.Verify the data loaded correctly by printing the shapes
print("Data loaded successfully!")
print(f"Training images shape:{train_images.shape}")
print(f"Training labels shape:{train_labels.shape}")
print(f"Testing images shape:{test_images.shape}")
print(f"Testing labels shape:{test_labels.shape}")

Data loaded successfully!
Training images shape:(50000, 32, 32, 3)
Training labels shape:(50000, 1)
Testing images shape:(10000, 32, 32, 3)
Testing labels shape:(10000, 1)


In [27]:
import torchvision
import torchvision.transforms as transforms

# --- Load CIFAR-10 Data in PyTorch ---

# 1. Define the transformations (convert to tensor and normalize to range [-1, 1])
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# 2. Download and load the Training data
trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=64, shuffle=True, num_workers=2)

# 3. Download and load the Testing data
testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=64, shuffle=False, num_workers=2)

# 4. Verify the data loaded correctly
print("PyTorch Data loaded successfully!")
print(f"Number of training batches: {len(trainloader)}")
print(f"Number of testing batches: {len(testloader)}")

# Optional: Check the shape of one batch of images
images, labels = next(iter(trainloader))
print(f"Shape of one batch of images: {images.shape}") # Should be torch.Size([64, 3, 32, 32])

PyTorch Data loaded successfully!
Number of training batches: 782
Number of testing batches: 157
Shape of one batch of images: torch.Size([64, 3, 32, 32])


In [29]:
# TF Simple NN
tf_simple_model = models.Sequential([
    layers.Flatten(input_shape=(32, 32, 3)),
    layers.Dense(128, activation='relu'),
    layers.Dense(10, activation='softmax')
])

tf_simple_model.compile(optimizer='adam',
                        loss='sparse_categorical_crossentropy',
                        metrics=['accuracy'])

print("Training TF Simple NN...")
start_time_tf_simple = time.time()
tf_simple_history = tf_simple_model.fit(train_images, train_labels, epochs=5, validation_split=0.1, verbose=1)
end_time_tf_simple = time.time()

tf_simple_time = end_time_tf_simple - start_time_tf_simple
tf_simple_acc = tf_simple_history.history['val_accuracy'][-1]
print(f"TF Simple NN Time: {tf_simple_time:.2f} seconds | Val Accuracy: {tf_simple_acc:.4f}")

/usr/local/lib/python3.13/dist-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Training TF Simple NN...
Epoch 1/5
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 15s 8ms/step - accuracy: 0.3162 - loss: 1.9112 - val_accuracy: 0.3798 - val_loss: 1.7693
Epoch 2/5
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 12s 8ms/step - accuracy: 0.3728 - loss: 1.7570 - val_accuracy: 0.3930 - val_loss: 1.7124
Epoch 3/5
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 12s 8ms/step - accuracy: 0.3914 - loss: 1.7006 - val_accuracy: 0.3956 - val_loss: 1.6791
Epoch 4/5
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 12s 8ms/step - accuracy: 0.4026 - loss: 1.6699 - val_accuracy: 0.3910 - val_loss: 1.6995
Epoch 5/5
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 12s 8ms/step - accuracy: 0.4099 - loss: 1.6496 - val_accuracy: 0.3928 - val_loss: 1.6932
TF Simple NN Time: 74.43 seconds | Val Accuracy: 0.3928


In [31]:
# TF CNN
tf_cnn_model = models.Sequential([
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(32, 32, 3)),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dense(10, activation='softmax')
])

tf_cnn_model.compile(optimizer='adam',
                     loss='sparse_categorical_crossentropy',
                     metrics=['accuracy'])

print("Training TF CNN...")
start_time_tf_cnn = time.time()
tf_cnn_history = tf_cnn_model.fit(train_images, train_labels, epochs=5, validation_split=0.1, verbose=1)
end_time_tf_cnn = time.time()

tf_cnn_time = end_time_tf_cnn - start_time_tf_cnn
tf_cnn_acc = tf_cnn_history.history['val_accuracy'][-1]
print(f"TF CNN Time: {tf_cnn_time:.2f} seconds | Val Accuracy: {tf_cnn_acc:.4f}")

/usr/local/lib/python3.13/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Training TF CNN...
Epoch 1/5
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 61s 43ms/step - accuracy: 0.4710 - loss: 1.4754 - val_accuracy: 0.5864 - val_loss: 1.1932
Epoch 2/5
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 82s 43ms/step - accuracy: 0.6152 - loss: 1.1017 - val_accuracy: 0.6382 - val_loss: 1.0454
Epoch 3/5
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 62s 44ms/step - accuracy: 0.6632 - loss: 0.9711 - val_accuracy: 0.6668 - val_loss: 0.9747
Epoch 4/5
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 60s 43ms/step - accuracy: 0.6923 - loss: 0.8868 - val_accuracy: 0.6814 - val_loss: 0.9377
Epoch 5/5
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 63s 45ms/step - accuracy: 0.7148 - loss: 0.8194 - val_accuracy: 0.6938 - val_loss: 0.9139
TF CNN Time: 330.27 seconds | Val Accuracy: 0.6938


In [33]:
class PtSimpleNN(nn.Module):
    def __init__(self):
        super(PtSimpleNN, self).__init__()
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(32 * 32 * 3, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = self.flatten(x)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

pt_simple_model = PtSimpleNN()
optimizer_simple = optim.Adam(pt_simple_model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

print("Training PyTorch Simple NN...")
start_time_pt_simple = time.time()
for epoch in range(5):
    for images, labels in trainloader:
        optimizer_simple.zero_grad()
        outputs = pt_simple_model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer_simple.step()
end_time_pt_simple = time.time()

# Evaluate PyTorch Simple NN
pt_simple_correct = 0
pt_simple_total = 0
with torch.no_grad():
    for images, labels in testloader:
        outputs = pt_simple_model(images)
        _, predicted = torch.max(outputs.data, 1)
        pt_simple_total += labels.size(0)
        pt_simple_correct += (predicted == labels).sum().item()

pt_simple_time = end_time_pt_simple - start_time_pt_simple
pt_simple_acc = pt_simple_correct / pt_simple_total
print(f"PyTorch Simple NN Time: {pt_simple_time:.2f} seconds | Test Accuracy: {pt_simple_acc:.4f}")

Training PyTorch Simple NN...
PyTorch Simple NN Time: 97.44 seconds | Test Accuracy: 0.5069


In [35]:
class PtCNN(nn.Module):
    def __init__(self):
        super(PtCNN, self).__init__()
        self.conv1 = nn.Conv2d(3, 32, 3)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(32, 64, 3)
        self.fc1 = nn.Linear(64 * 6 * 6, 64) # 32x32 -> pool -> 14x14 -> pool -> 6x6
        self.fc2 = nn.Linear(64, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(-1, 64 * 6 * 6)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

pt_cnn_model = PtCNN()
optimizer_cnn = optim.Adam(pt_cnn_model.parameters(), lr=0.001)

print("Training PyTorch CNN...")
start_time_pt_cnn = time.time()
for epoch in range(5):
    for images, labels in trainloader:
        optimizer_cnn.zero_grad()
        outputs = pt_cnn_model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer_cnn.step()
end_time_pt_cnn = time.time()

# Evaluate PyTorch CNN
pt_cnn_correct = 0
pt_cnn_total = 0
with torch.no_grad():
    for images, labels in testloader:
        outputs = pt_cnn_model(images)
        _, predicted = torch.max(outputs.data, 1)
        pt_cnn_total += labels.size(0)
        pt_cnn_correct += (predicted == labels).sum().item()

pt_cnn_time = end_time_pt_cnn - start_time_pt_cnn
pt_cnn_acc = pt_cnn_correct / pt_cnn_total
print(f"PyTorch CNN Time: {pt_cnn_time:.2f} seconds | Test Accuracy: {pt_cnn_acc:.4f}")

Training PyTorch CNN...
PyTorch CNN Time: 354.94 seconds | Test Accuracy: 0.6938
